# 🛡️ SpamGuard AI — Model Development & Evaluation Notebook
**Project:** Swynex Technologies AI Final Task  
**Author:** Harshadvali Shaik  
**Objective:** Build, train, evaluate, and export an explainable SMS Spam & Phishing Classification model using TF-IDF and Logistic Regression.


## 1. Environment & Library Setup

In [ ]:
import re
import unicodedata
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

print('Libraries loaded successfully!')


## 2. Load & Inspect Dataset

In [ ]:
df = pd.read_csv('../data/cleaned_dataset.csv')
print(f'Total records: {len(df)}')
display(df.head())
print('\nClass distribution:')
print(df['label'].value_counts())


## 3. Exploratory Data Analysis (EDA)

In [ ]:
df['char_count'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print('Character length statistics:')
display(df.groupby('label')[['char_count', 'word_count']].mean())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['label'].value_counts().plot(kind='bar', color=['#3b82f6', '#ef4444'])
plt.title('Class Balance (Ham vs Spam)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='char_count', hue='label', bins=40, kde=True)
plt.title('Message Length Distribution')
plt.xlim(0, 300)
plt.tight_layout()
plt.show()


## 4. Text Preprocessing & Cleaning

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' <EMAIL> ', text)
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', ' <PHONE> ', text)
    text = re.sub(r'[\$£€¥₹]\s*\d+(?:[.,]\d+)?', ' <CURRENCY> ', text)
    text = re.sub(r'([!?.,]){2,}', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
df['target'] = (df['label'] == 'spam').astype(int)
print('Sample cleaned message:')
print('Raw:    ', df['text'].iloc[3])
print('Cleaned:', df['cleaned_text'].iloc[3])


## 5. Train / Test Data Split (80 / 20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['target'],
    test_size=0.20, random_state=42, stratify=df['target']
)
print(f'Training samples: {len(X_train)} (Spam: {y_train.sum()})')
print(f'Testing samples:  {len(X_test)} (Spam: {y_test.sum()})')


## 6. Model Pipeline Construction & Training

In [ ]:
union = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('char_tfidf', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3, sublinear_tf=True))
])

pipeline = Pipeline([
    ('union', union),
    ('clf', LogisticRegression(C=3.0, class_weight='balanced', max_iter=1000, random_state=42, solver='liblinear'))
])

pipeline.fit(X_train, y_train)
print('Pipeline training complete!')


## 7. Model Evaluation on Holdout Test Set

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f'Accuracy:  {acc * 100:.2f}%')
print(f'Precision: {prec * 100:.2f}%')
print(f'Recall:    {rec * 100:.2f}%')
print(f'F1 Score:  {f1 * 100:.2f}%')
print(f'ROC-AUC:   {auc * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate (Ham)', 'Spam']))


## 8. Confusion Matrix & ROC Curve Visualizations

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Test Set Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.4f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 9. Save Trained Model Pipeline

In [ ]:
joblib.dump(pipeline, '../models/model.pkl')
print('Model successfully persisted to ../models/model.pkl!')
